In [1]:
# If MNE is not installed, run this once in a separate notebook cell:
# %pip install mne

In [2]:

import os

import mne
import numpy as np
import scipy
import torch

from sklearn.discriminant_analysis import _cov
from sklearn.utils import shuffle
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

n_ses = 4

seed = 20200220
re_sfreq = 250

tmin = -0.2
tmax = 1.0

whiten = True
mvnn_dim = "epochs"

save_root = "preprocessed_data"

# Expected EEG channel order
chan_order = [
    "Fp1", "Fp2", "AF7", "AF3", "AFz", "AF4", "AF8",
    "F7", "F5", "F3", "F1", "F2", "F4", "F6", "F8",
    "FT9", "FT7", "FC5", "FC3", "FC1", "FCz", "FC2",
    "FC4", "FC6", "FT8", "FT10", "T7", "C5", "C3",
    "C1", "Cz", "C2", "C4", "C6", "T8", "TP9",
    "TP7", "CP5", "CP3", "CP1", "CPz", "CP2", "CP4",
    "CP6", "TP8", "TP10", "P7", "P5", "P3", "P1",
    "Pz", "P2", "P4", "P6", "P8", "PO7", "PO3",
    "POz", "PO4", "PO8", "O1", "Oz", "O2",
]


# ---------------------------------------------------------------------------
# MVNN whitening
# ---------------------------------------------------------------------------

def mvnn(epoched_test, epoched_train):
    """
    Apply multivariate noise normalisation separately for each session.

    The covariance matrix is calculated only from the training data.
    The resulting whitening matrix is applied to both training and test data.
    """

    whitened_test = []
    whitened_train = []

    for s in range(n_ses):

        print(f"\nMVNN session {s + 1}/{n_ses}")

        session_data = [
            epoched_test[s],
            epoched_train[s],
        ]

        # Shape:
        # data partition × channels × channels
        sigma_part = np.empty(
            (
                len(session_data),
                session_data[0].shape[2],
                session_data[0].shape[2],
            )
        )

        for p in range(sigma_part.shape[0]):

            # Shape:
            # image condition × channels × channels
            sigma_cond = np.empty(
                (
                    session_data[p].shape[0],
                    session_data[0].shape[2],
                    session_data[0].shape[2],
                )
            )

            for i in tqdm(range(session_data[p].shape[0])):

                cond_data = session_data[p][i]

                if mvnn_dim == "time":

                    sigma_cond[i] = np.mean(
                        [
                            _cov(
                                cond_data[:, :, t],
                                shrinkage="auto",
                            )
                            for t in range(cond_data.shape[2])
                        ],
                        axis=0,
                    )

                elif mvnn_dim == "epochs":

                    sigma_cond[i] = np.mean(
                        [
                            _cov(
                                np.transpose(cond_data[e]),
                                shrinkage="auto",
                            )
                            for e in range(cond_data.shape[0])
                        ],
                        axis=0,
                    )

                else:
                    raise ValueError(
                        "mvnn_dim must be either 'time' or 'epochs'."
                    )

            # Average across image conditions
            sigma_part[p] = sigma_cond.mean(axis=0)

        # Use only training data to calculate the whitening matrix.
        # session_data[0] is test and session_data[1] is training.
        sigma_tot = sigma_part[1]

        sigma_inv = scipy.linalg.fractional_matrix_power(
            sigma_tot,
            -0.5,
        )

        # Numerical computation can sometimes leave a negligible imaginary part.
        sigma_inv = np.real_if_close(
            sigma_inv,
            tol=1000,
        )

        if np.iscomplexobj(sigma_inv):
            raise ValueError(
                f"MVNN matrix for session {s + 1} contains "
                "non-negligible complex values."
            )

        # Whiten test data
        current_test = np.reshape(
            (
                np.reshape(
                    session_data[0],
                    (
                        -1,
                        session_data[0].shape[2],
                        session_data[0].shape[3],
                    ),
                )
                .swapaxes(1, 2)
                @ sigma_inv
            ).swapaxes(1, 2),
            session_data[0].shape,
        )

        # Whiten training data
        current_train = np.reshape(
            (
                np.reshape(
                    session_data[1],
                    (
                        -1,
                        session_data[1].shape[2],
                        session_data[1].shape[3],
                    ),
                )
                .swapaxes(1, 2)
                @ sigma_inv
            ).swapaxes(1, 2),
            session_data[1].shape,
        )

        current_test = current_test.astype(np.float32)
        current_train = current_train.astype(np.float32)

        if not np.isfinite(current_test).all():
            raise ValueError(
                f"Test data contains NaN or infinity "
                f"after MVNN in session {s + 1}."
            )

        if not np.isfinite(current_train).all():
            raise ValueError(
                f"Training data contains NaN or infinity "
                f"after MVNN in session {s + 1}."
            )

        whitened_test.append(current_test)
        whitened_train.append(current_train)

    return whitened_test, whitened_train


# ---------------------------------------------------------------------------
# Epoch raw EEG
# ---------------------------------------------------------------------------

def epoch_data(mode, sub):
    """
    Load, epoch, baseline-correct and resample EEG data.

    Output for each session:
        image conditions × repetitions × channels × 250 time points
    """

    if mode not in {"train", "test"}:
        raise ValueError("mode must be 'train' or 'test'.")

    epoched_data = []
    img_conditions = []

    final_ch_names = None
    final_times = None

    for s in range(n_ses):

        eeg_path = os.path.join(
            raw_root,
            f"sub-{sub:02d}",
            f"ses-{s + 1:02d}",
            f"raw_eeg_{mode}.npy",
        )

        print("\nLoading:", eeg_path)

        if not os.path.isfile(eeg_path):
            raise FileNotFoundError(
                f"EEG file was not found: {eeg_path}"
            )

        eeg_dict = np.load(
            eeg_path,
            allow_pickle=True,
        ).item()

        ch_names = list(eeg_dict["ch_names"])
        sfreq = float(eeg_dict["sfreq"])
        ch_types = list(eeg_dict["ch_types"])
        eeg_array = eeg_dict["raw_eeg_data"]

        print("Raw shape:", eeg_array.shape)
        print("Original sampling frequency:", sfreq)

        # Convert continuous EEG to an MNE Raw object.
        info = mne.create_info(
            ch_names=ch_names,
            sfreq=sfreq,
            ch_types=ch_types,
        )

        raw = mne.io.RawArray(
            eeg_array,
            info,
        )

        # Find events from the stimulus channel.
        events = mne.find_events(
            raw,
            stim_channel="stim",
        )

        print("Events before target removal:", len(events))

        # Remove target trials with event ID 99999.
        idx_target = np.where(
            events[:, 2] == 99999
        )[0]

        events = np.delete(
            events,
            idx_target,
            axis=0,
        )

        print("Events after target removal:", len(events))

        # Remove the stimulus channel and place EEG channels
        # in the specified order.
        raw.pick_channels(
            chan_order,
            ordered=True,
        )

        # Create epochs from -200 ms to 1000 ms.
        # Baseline correction uses the pre-stimulus interval.
        epochs = mne.Epochs(
            raw,
            events,
            tmin=tmin,
            tmax=tmax,
            baseline=(None, 0),
            preload=True,
        )

        # Downsample from 1000 Hz to 250 Hz.
        if re_sfreq < sfreq:
            epochs.resample(re_sfreq)

        actual_sfreq = float(
            epochs.info["sfreq"]
        )

        if not np.isclose(
            actual_sfreq,
            re_sfreq,
        ):
            raise ValueError(
                f"Expected {re_sfreq} Hz but obtained "
                f"{actual_sfreq} Hz."
            )

        ch_names = list(
            epochs.info["ch_names"]
        )

        if ch_names != chan_order:
            raise ValueError(
                "Final EEG channel order does not match chan_order."
            )

        data = epochs.get_data()
        event_labels = epochs.events[:, 2]
        img_cond = np.unique(event_labels)

        if mode == "test":
            max_rep = 20
        else:
            max_rep = 2

        # Shape:
        # image conditions × repetitions × channels × time points
        sorted_data = np.zeros(
            (
                len(img_cond),
                max_rep,
                data.shape[1],
                data.shape[2],
            ),
            dtype=np.float32,
        )

        for i in range(len(img_cond)):

            idx = np.where(
                event_labels == img_cond[i]
            )[0]

            if len(idx) < max_rep:
                raise ValueError(
                    f"{mode}, session {s + 1}, condition "
                    f"{img_cond[i]} has only {len(idx)} repetitions. "
                    f"{max_rep} repetitions are required."
                )

            idx = shuffle(
                idx,
                random_state=seed,
                n_samples=max_rep,
            )

            sorted_data[i] = data[idx].astype(
                np.float32
            )

        # The paper code keeps the final 250 time points,
        # corresponding to approximately one second after stimulus onset.
        processed_data = sorted_data[
            :,
            :,
            :,
            -re_sfreq:,
        ]

        # Save only the time values corresponding to the retained EEG data.
        selected_times = epochs.times[
            -re_sfreq:
        ]

        if processed_data.shape[-1] != re_sfreq:
            raise ValueError(
                f"Expected {re_sfreq} time points, but obtained "
                f"{processed_data.shape[-1]}."
            )

        if len(selected_times) != re_sfreq:
            raise ValueError(
                f"Expected {re_sfreq} time values, but obtained "
                f"{len(selected_times)}."
            )

        if not np.isfinite(processed_data).all():
            raise ValueError(
                f"{mode} session {s + 1} contains "
                "NaN or infinite EEG values."
            )

        print(
            f"{mode}, session {s + 1}:",
            processed_data.shape,
        )

        print(
            "Saved time range:",
            f"{selected_times[0]:.3f} to "
            f"{selected_times[-1]:.3f} seconds",
        )

        if final_ch_names is None:
            final_ch_names = ch_names

        elif final_ch_names != ch_names:
            raise ValueError(
                "Channel order differs between sessions."
            )

        if final_times is None:
            final_times = selected_times

        elif not np.allclose(
            final_times,
            selected_times,
        ):
            raise ValueError(
                "Time vectors differ between sessions."
            )

        epoched_data.append(
            processed_data
        )

        img_conditions.append(
            img_cond
        )

    return (
        epoched_data,
        img_conditions,
        final_ch_names,
        final_times,
    )


# ---------------------------------------------------------------------------
# Load and epoch test and training data for each subject
# ---------------------------------------------------------------------------

for sub in range(2, 11):

    raw_root = f"sub{sub:02d}_raw"

    if whiten:
        save_dir = os.path.join(
            save_root,
            f"Preprocessed_data_{re_sfreq}Hz_whiten",
            f"sub-{sub:02d}",
        )
    else:
        save_dir = os.path.join(
            save_root,
            f"Preprocessed_data_{re_sfreq}Hz_no_whiten",
            f"sub-{sub:02d}",
        )

    os.makedirs(save_dir, exist_ok=True)

    print(f"\n=== PROCESSING SUB-{sub:02d} ===")
    print("Raw data:", raw_root)
    print("Saving to:", save_dir)


    print("\n=== EPOCHING TEST DATA ===")

    (
        eeg_test,
        img_conditions_test,
        test_ch_names,
        test_times,
    ) = epoch_data(
        "test",
        sub,
    )


    print("\n=== EPOCHING TRAINING DATA ===")

    (
        eeg_train,
        img_conditions_train,
        train_ch_names,
        train_times,
    ) = epoch_data(
        "train",
        sub,
    )


    if test_ch_names != train_ch_names:
        raise ValueError(
            "Training and test channel orders do not match."
        )

    if not np.allclose(
        test_times,
        train_times,
    ):
        raise ValueError(
            "Training and test time vectors do not match."
        )

    ch_names = train_ch_names
    times = train_times


    # ---------------------------------------------------------------------------
    # Apply MVNN
    # ---------------------------------------------------------------------------

    if whiten:

        print("\n=== APPLYING MVNN ===")

        whitened_test, whitened_train = mvnn(
            eeg_test,
            eeg_train,
        )

        del eeg_test
        del eeg_train

    else:

        whitened_test = eeg_test
        whitened_train = eeg_train


    # ---------------------------------------------------------------------------
    # Merge test sessions
    # ---------------------------------------------------------------------------

    print("\n=== MERGING TEST DATA ===")

    session_list = np.zeros(
        (200, 80),
        dtype=np.int64,
    )

    for s in range(n_ses):

        if s == 0:
            merged_test = whitened_test[s]

        else:
            merged_test = np.append(
                merged_test,
                whitened_test[s],
                axis=1,
            )

        start_index = (
            merged_test.shape[1]
            - whitened_test[s].shape[1]
        )

        end_index = merged_test.shape[1]

        session_list[
            :,
            start_index:end_index
        ] = s


    del whitened_test


    print(
        "Test before repetition averaging:",
        merged_test.shape,
    )

    # Average all 80 repetitions of each test image.
    merged_test = merged_test.mean(
        axis=1,
        dtype=np.float32,
    )

    print(
        "Test after repetition averaging:",
        merged_test.shape,
    )


    if merged_test.shape != (
        200,
        63,
        re_sfreq,
    ):
        raise ValueError(
            "Unexpected averaged test EEG shape: "
            f"{merged_test.shape}"
        )


    # ---------------------------------------------------------------------------
    # Load test image metadata
    # ---------------------------------------------------------------------------

    test_img_directory = "images/test_images"

    if not os.path.isdir(test_img_directory):
        raise FileNotFoundError(
            f"Test image directory was not found: "
            f"{test_img_directory}"
        )


    all_folders = [
        folder
        for folder in os.listdir(test_img_directory)
        if os.path.isdir(
            os.path.join(
                test_img_directory,
                folder,
            )
        )
    ]

    all_folders.sort()


    test_images = []
    test_labels = []
    test_texts = []


    for label, folder in enumerate(all_folders):

        folder_path = os.path.join(
            test_img_directory,
            folder,
        )

        all_images = [
            image_name
            for image_name in os.listdir(folder_path)
            if image_name.lower().endswith(
                (
                    ".png",
                    ".jpg",
                    ".jpeg",
                )
            )
        ]

        all_images.sort()

        test_images.extend(
            os.path.join(
                folder_path,
                image_name,
            )
            for image_name in all_images
        )

        test_labels.extend(
            [label] * len(all_images)
        )

        test_texts.extend(
            image_name.rsplit(
                "_",
                1,
            )[0]
            for image_name in all_images
        )


    test_labels = np.asarray(
        test_labels,
        dtype=np.int64,
    )


    if len(test_images) != merged_test.shape[0]:
        raise ValueError(
            f"Found {len(test_images)} test images, but EEG contains "
            f"{merged_test.shape[0]} image conditions."
        )

    if len(test_labels) != merged_test.shape[0]:
        raise ValueError(
            "Test label count does not match test EEG."
        )

    if len(test_texts) != merged_test.shape[0]:
        raise ValueError(
            "Test text count does not match test EEG."
        )


    print("Test EEG shape:", merged_test.shape)
    print("Test labels:", test_labels.shape)
    print("Test images:", len(test_images))
    print("Test texts:", len(test_texts))


    # ---------------------------------------------------------------------------
    # Save averaged test data
    # ---------------------------------------------------------------------------

    test_dict = {
        "eeg": torch.from_numpy(
            merged_test
        ).to(torch.float32),

        "label": torch.from_numpy(
            test_labels
        ).to(torch.long),

        "img": list(test_images),

        "text": list(test_texts),

        "ch_names": list(ch_names),

        "times": torch.as_tensor(
            times,
            dtype=torch.float32,
        ),

        "sfreq": float(re_sfreq),
    }


    test_path = os.path.join(
        save_dir,
        "test.pt",
    )


    torch.save(
        test_dict,
        test_path,
    )


    print("Saved test data:", test_path)


    # ---------------------------------------------------------------------------
    # Merge training sessions
    # ---------------------------------------------------------------------------

    print("\n=== MERGING TRAINING DATA ===")


    ses_list = np.zeros(
        (33080, 2),
        dtype=np.int64,
    )


    for s in range(n_ses):

        if s == 0:

            white_data = whitened_train[s]
            img_cond = img_conditions_train[s]

        else:

            white_data = np.append(
                white_data,
                whitened_train[s],
                axis=0,
            )

            img_cond = np.append(
                img_cond,
                img_conditions_train[s],
                axis=0,
            )

        start_index = (
            white_data.shape[0]
            - whitened_train[s].shape[0]
        )

        end_index = white_data.shape[0]

        ses_list[
            start_index:end_index
        ] = s


    del whitened_train


    print("ses_list:", ses_list.shape)


    # Shape before averaging:
    # image conditions × 4 repetitions × channels × time points
    merged_train = np.zeros(
        (
            len(np.unique(img_cond)),
            white_data.shape[1] * 2,
            white_data.shape[2],
            white_data.shape[3],
        ),
        dtype=np.float32,
    )


    sorted_session_list = np.zeros(
        (
            len(np.unique(img_cond)),
            4,
        ),
        dtype=np.int64,
    )


    for i in range(len(np.unique(img_cond))):

        idx = np.where(
            img_cond == i + 1
        )[0]

        if len(idx) != 2:
            raise ValueError(
                f"Training image condition {i + 1} appears in "
                f"{len(idx)} sessions instead of 2."
            )

        for r in range(len(idx)):

            sorted_session_list[
                i,
                r * 2:r * 2 + 2
            ] = ses_list[idx[r]]

            if r == 0:

                ordered_data = white_data[
                    idx[r]
                ]

            else:

                ordered_data = np.append(
                    ordered_data,
                    white_data[idx[r]],
                    axis=0,
                )

        if ordered_data.shape[0] != 4:
            raise ValueError(
                f"Training image condition {i + 1} has "
                f"{ordered_data.shape[0]} repetitions instead of 4."
            )

        merged_train[i] = ordered_data


    del ordered_data
    del white_data


    print(
        "Training before repetition averaging:",
        merged_train.shape,
    )

    # Average all four repetitions of each training image.
    merged_train = merged_train.mean(
        axis=1,
        dtype=np.float32,
    )

    print(
        "Training after repetition averaging:",
        merged_train.shape,
    )


    if merged_train.shape != (
        16540,
        63,
        re_sfreq,
    ):
        raise ValueError(
            "Unexpected averaged training EEG shape: "
            f"{merged_train.shape}"
        )


    # ---------------------------------------------------------------------------
    # Load training image metadata
    # ---------------------------------------------------------------------------

    train_img_directory = "images/training_images"

    if not os.path.isdir(train_img_directory):
        raise FileNotFoundError(
            f"Training image directory was not found: "
            f"{train_img_directory}"
        )


    all_folders = [
        folder
        for folder in os.listdir(train_img_directory)
        if os.path.isdir(
            os.path.join(
                train_img_directory,
                folder,
            )
        )
    ]

    all_folders.sort()


    train_images = []
    train_labels = []
    train_texts = []


    for label, folder in enumerate(all_folders):

        folder_path = os.path.join(
            train_img_directory,
            folder,
        )

        all_images = [
            image_name
            for image_name in os.listdir(folder_path)
            if image_name.lower().endswith(
                (
                    ".png",
                    ".jpg",
                    ".jpeg",
                )
            )
        ]

        all_images.sort()

        train_images.extend(
            os.path.join(
                folder_path,
                image_name,
            )
            for image_name in all_images
        )

        train_labels.extend(
            [label] * len(all_images)
        )

        train_texts.extend(
            image_name.rsplit(
                "_",
                1,
            )[0]
            for image_name in all_images
        )


    train_labels = np.asarray(
        train_labels,
        dtype=np.int64,
    )


    if len(train_images) != merged_train.shape[0]:
        raise ValueError(
            f"Found {len(train_images)} training images, but EEG contains "
            f"{merged_train.shape[0]} image conditions."
        )

    if len(train_labels) != merged_train.shape[0]:
        raise ValueError(
            "Training label count does not match training EEG."
        )

    if len(train_texts) != merged_train.shape[0]:
        raise ValueError(
            "Training text count does not match training EEG."
        )


    print("Training EEG shape:", merged_train.shape)
    print("Training labels:", train_labels.shape)
    print("Training images:", len(train_images))
    print("Training texts:", len(train_texts))


    # ---------------------------------------------------------------------------
    # Save averaged training data
    # ---------------------------------------------------------------------------

    train_dict = {
        "eeg": torch.from_numpy(
            merged_train
        ).to(torch.float32),

        "label": torch.from_numpy(
            train_labels
        ).to(torch.long),

        "img": list(train_images),

        "text": list(train_texts),

        "ch_names": list(ch_names),

        "times": torch.as_tensor(
            times,
            dtype=torch.float32,
        ),

        "sfreq": float(re_sfreq),
    }


    train_path = os.path.join(
        save_dir,
        "train.pt",
    )


    torch.save(
        train_dict,
        train_path,
    )


    print("Saved training data:", train_path)


    # ---------------------------------------------------------------------------
    # Final loading check
    # ---------------------------------------------------------------------------

    print("\n=== CHECKING SAVED FILES ===")


    loaded_train = torch.load(
        train_path,
        map_location="cpu",
        weights_only=False,
    )

    loaded_test = torch.load(
        test_path,
        map_location="cpu",
        weights_only=False,
    )


    print("Train keys:", list(loaded_train.keys()))
    print("Train EEG:", loaded_train["eeg"].shape)
    print("Train labels:", loaded_train["label"].shape)
    print("Train times:", loaded_train["times"].shape)
    print("Train sampling frequency:", loaded_train["sfreq"])


    print("Test keys:", list(loaded_test.keys()))
    print("Test EEG:", loaded_test["eeg"].shape)
    print("Test labels:", loaded_test["label"].shape)
    print("Test times:", loaded_test["times"].shape)
    print("Test sampling frequency:", loaded_test["sfreq"])


    assert loaded_train["eeg"].shape == (
        16540,
        63,
        250,
    )

    assert loaded_test["eeg"].shape == (
        200,
        63,
        250,
    )

    assert loaded_train["times"].shape[0] == 250
    assert loaded_test["times"].shape[0] == 250

    assert torch.isfinite(
        loaded_train["eeg"]
    ).all()

    assert torch.isfinite(
        loaded_test["eeg"]
    ).all()


    print("\nPREPROCESSING COMPLETED SUCCESSFULLY")


=== PROCESSING SUB-02 ===
Raw data: sub02_raw
Saving to: preprocessed_data/Preprocessed_data_250Hz_whiten/sub-02

=== EPOCHING TEST DATA ===

Loading: sub02_raw/sub-02/ses-01/raw_eeg_test.npy


Raw shape: (64, 1427880)
Original sampling frequency: 1000.0


Creating RawArray with float64 data, n_channels=64, n_times=1427880


    Range : 0 ... 1427879 =      0.000 ...  1427.879 secs


Ready.


...

Training before repetition averaging: (16540, 4, 63, 250)


Training after repetition averaging: (16540, 63, 250)
Training EEG shape: (16540, 63, 250)
Training labels: (16540,)
Training images: 16540
Training texts: 16540


Saved training data: preprocessed_data/Preprocessed_data_250Hz_whiten/sub-10/train.pt

=== CHECKING SAVED FILES ===


Train keys: ['eeg', 'label', 'img', 'text', 'ch_names', 'times', 'sfreq']
Train EEG: torch.Size([16540, 63, 250])
Train labels: torch.Size([16540])
Train times: torch.Size([250])
Train sampling frequency: 250.0
Test keys: ['eeg', 'label', 'img', 'text', 'ch_names', 'times', 'sfreq']
Test EEG: torch.Size([200, 63, 250])
Test labels: torch.Size([200])
Test times: torch.Size([250])
Test sampling frequency: 250.0



PREPROCESSING COMPLETED SUCCESSFULLY
